# Hands-on: The Traveling Salesman Problem with GAMSPy

**Goal:** find the shortest round trip that visits each of 12 German cities exactly once and returns to the start.

We will:

1. Load city data and build a distance table
2. Write and solve the **assignment problem** (every city has one way in and one way out)
3. See why that is not enough (**subtours**) and fix it with an **iterative subtour-elimination (cutting-plane) loop**
4. *(Bonus)* Solve the compact **MTZ** formulation

Based on GAMS Model Library model [TSP4](https://www.gams.com/latest/gamslib_ml/libhtml/gamslib_tsp4.html).

> Cells marked **`TODO`** are for you. Replace `...` with your code, then run the cell. Some are followed by a **CHECK** cell that tells you if your answer is right.

## 0. Setup

If you are on Google Colab, run the install cell first. The next cell downloads `cities.json` from the [workshop repo](https://github.com/boxblox/csmio_26/tree/main/workshop/data).

In [ ]:
# Uncomment on Colab / fresh environments
# !pip install gamspy networkx scipy matplotlib pandas

In [ ]:
# Download the data files (works on Google Colab and locally)
import os
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/boxblox/csmio_26/main/workshop/data/"
for fname in ['cities.json']:
    if not os.path.exists(fname):
        urllib.request.urlretrieve(DATA_URL + fname, fname)
    print("ok:", fname)

In [ ]:
from __future__ import annotations

import json

import gamspy as gp
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from matplotlib.patches import Rectangle
from scipy.spatial import ConvexHull


# ---------- plotting helpers (no need to edit) ----------
def shade_conv_hull(ax, pos, nodes):
    points = np.array([pos[n] for n in nodes])
    if len(nodes) >= 3:
        hull = ConvexHull(points)
        ax.add_patch(plt.Polygon(points[hull.vertices], closed=True, fill=True,
                                 facecolor="lightblue", alpha=0.2,
                                 edgecolor="blue", linewidth=2))
    elif len(nodes) == 2:
        p1, p2 = points
        pad = 0.2
        (x0, y0), (x1, y1) = np.minimum(p1, p2) - pad, np.maximum(p1, p2) + pad
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, facecolor="lightblue",
                               alpha=0.2, edgecolor="blue", linewidth=2))


def plot_cities(pos, complete=False, title="TSP: 12 cities"):
    fig, ax = plt.subplots(figsize=(7, 6))
    if complete:
        pts = list(pos.values())
        for a in range(len(pts)):
            for b in range(a + 1, len(pts)):
                ax.plot([pts[a][0], pts[b][0]], [pts[a][1], pts[b][1]],
                        color="grey", linewidth=0.4)
    for name, (lon, lat) in pos.items():
        ax.scatter(lon, lat, color="red", zorder=3)
        ax.text(lon + 0.08, lat + 0.08, name, fontsize=9)
    ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def plot_solution(edges, step_idx, nodes, pos, S, title=None):
    G = nx.DiGraph()
    G.add_nodes_from(nodes)
    G.add_edges_from(edges)
    fig, ax = plt.subplots(figsize=(7, 6))
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color="lightblue", node_size=150)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=9)
    nx.draw_networkx_edges(G, pos, ax=ax, edgelist=edges, edge_color="red", width=2)
    for s in S:
        shade_conv_hull(ax, pos, s.nodes())
    ax.set_title(title or f"TSP solution - iteration {step_idx + 1}  ({len(S)} subtour(s))")
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def current_tours(x, nodes):
    """Read the solution of x and return (edges, list of strongly connected components)."""
    G = nx.DiGraph()
    G.add_nodes_from(nodes)
    sel = x.records[x.records["level"] > 0.5]
    G.add_edges_from(zip(sel.iloc[:, 0], sel.iloc[:, 1]))
    S = [G.subgraph(comp).copy() for comp in nx.strongly_connected_components(G)]
    return list(G.edges()), S

## 1. Data

We pick 12 cities from `cities.json` and use the straight-line (Euclidean) distance between their (latitude, longitude) points as the travel cost.

You will use two tables later:

* `city_df`: one row per city, with columns `city`, `latitude`, `longitude`
* `distance_df`: one row per pair of cities, with columns `from_city`, `to_city`, `distance`. This is already in the format GAMSPy needs for the distance parameter $c_{ij}$: the index columns first, the value last.

In [ ]:
with open("cities.json") as fp:
    city_data = json.load(fp)

# one row per city, with simple column names
city_df = pd.json_normalize(city_data).rename(
    columns={"row.city": "city", "row.latitude": "latitude", "row.longitude": "longitude"}
)
city_df = city_df.sample(n=12, random_state=200)[["city", "latitude", "longitude"]]  # 12 cities, reproducible
city_df = city_df.reset_index(drop=True)

# distance between every pair of cities
coords = city_df[["latitude", "longitude"]].to_numpy()
dist = np.sqrt(((coords[:, np.newaxis, :] - coords[np.newaxis, :, :]) ** 2).sum(axis=-1))

# long format: from_city, to_city, distance  (a city to itself is left out)
distance_df = (
    pd.DataFrame(dist, index=city_df["city"], columns=city_df["city"])
    .rename_axis(index="from_city", columns="to_city")
    .stack()
    .rename("distance")
    .reset_index()
)
distance_df = distance_df[distance_df["from_city"] != distance_df["to_city"]].reset_index(drop=True)

# map positions for plotting: city -> (longitude, latitude)
pos = {r.city: (r.longitude, r.latitude) for r in city_df.itertuples()}

In [ ]:
print("city_df:", len(city_df), "cities")
display(city_df)

In [ ]:
print("distance_df:", len(distance_df), "pairs (first 15 shown)")
display(distance_df.head(15))

In [ ]:
plot_cities(pos)
plot_cities(pos, complete=True, title="Complete graph: 12 x 11 = 132 possible legs")

## 2. The assignment problem

**Data**

* $I$: the set of cities. We use $i$ for the city we leave and $j$ for the city we go to.
* $c_{ij}$: distance from city $i$ to city $j$ (from `distance_df`)

**Decision variable**

* $x_{ij} \in \{0, 1\}$: equals $1$ if the tour goes directly from city $i$ to city $j$, otherwise $0$

**Model**

$$
\begin{aligned}
\min \quad & \sum_{i \in I} \sum_{j \in I} c_{ij}\, x_{ij} && \text{(total distance)}\\
\text{s.t.}\quad & \sum_{j \in I} x_{ij} = 1 && \forall i \in I \qquad \text{(leave every city exactly once)}\\
& \sum_{i \in I} x_{ij} = 1 && \forall j \in I \qquad \text{(enter every city exactly once)}\\
& x_{ii} = 0 && \forall i \in I \qquad \text{(no leg from a city to itself)}\\
& x_{ij} \in \{0,1\} && \forall i, j \in I
\end{aligned}
$$

Every TSP tour satisfies these constraints, but not everything that satisfies them is a tour. We will see why after we solve it.

### TODO 1 - Container
A `gp.Container()` holds every set, parameter, variable, equation and model. Create one and call it `m`.

In [ ]:
# TODO: your code here
m = ...

### TODO 2 - Sets
Create the set of cities `i`, and an alias `j` of it, so we can talk about the *from*-city and the *to*-city.

*Hints:*
* `gp.Set(container, name, records=..., description=...)`. `records` can be a Python list or a pandas column, for example `city_df["city"]`.
* `gp.Alias(container, name, alias_with=...)` makes a second name for an existing set.

In [ ]:
# TODO: your code here
i = gp.Set(m, name="i", records=..., description="cities")
j = gp.Alias(m, name="j", alias_with=...)

In [ ]:
# CHECK
assert len(i.toList()) == 12, "i should contain the 12 cities"
print("Sets OK:", i.toList())

The distance parameter $c_{ij}$ and the decision variable $x_{ij}$ are given. `distance_df` goes straight into `records`, because its columns are already `from_city, to_city, distance`.

In [ ]:
c = gp.Parameter(m, name="c", domain=[i, j], records=distance_df, description="distance from i to j")
x = gp.Variable(m, name="x", type="binary", domain=[i, j], description="1 if we travel from i to j")

c.pivot().round(2)

### TODO 3 - Leaving and entering constraints
`rowsum` is declared for you.

* Declare `colsum` the same way, over the domain `j`.
* Write both equations:
  * $\sum_j x_{ij} = 1 \;\; \forall i$ (leave every city once)
  * $\sum_i x_{ij} = 1 \;\; \forall j$ (enter every city once)

*Hint:* `gp.Sum(j, x[i, j])` sums over `j`. Use `==` for "equals".

In [ ]:
rowsum = gp.Equation(m, name="rowsum", domain=i, description="leave each city exactly once")

# TODO: your code here
colsum = ...

rowsum[i] = ...
colsum[j] = ...

### TODO 4 - No leg from a city to itself
Fix the diagonal of `x` to 0, so that $x_{ii} = 0$.

*Hint:* `x.fx[...]` fixes a variable. Using the same set twice, `x.fx[i, i]`, picks only the diagonal.

In [ ]:
# TODO: your code here
x.fx[...] = ...

### TODO 5 - Objective
Total distance travelled: $\sum_{i}\sum_{j} c_{ij}\, x_{ij}$

*Hint:* to sum over two sets at once, pass a list: `gp.Sum([i, j], ...)`.

In [ ]:
# TODO: your code here
objective = ...

### TODO 6 - Model
Put the pieces together into a model called `assign`.

*Hint:* `gp.Model(container, name, equations=[...], problem=..., sense=..., objective=...)`
* `problem`: the type of model. Here `"MIP"`, because `x` is binary.
* `sense`: `"min"` or `"max"`

In [ ]:
# TODO: your code here
assign = gp.Model(m, name="assign", equations=..., problem=..., sense=..., objective=...)

### TODO 7 - Solve
Solve `assign`. The option `relative_optimality_gap=0` asks the solver for the proven best solution, not just a good one.

*Hint:* every model has a `.solve(...)` method.

In [ ]:
# TODO: your code here
...(options=gp.Options(relative_optimality_gap=0))

### TODO 8 - Look at the result
Print the objective value and the legs that are used ($x_{ij} = 1$).

*Hints:*
* `assign.objective_value` is the optimal total distance.
* `x.records` is a DataFrame with one row per $(i, j)$. Its `level` column holds the solution value.

In [ ]:
# TODO: your code here
print("Total distance:", ...)

legs = x.records[...]
legs[["i", "j", "level"]]

In [ ]:
# CHECK
assert abs(assign.objective_value - 14.9899) < 1e-3, f"got {assign.objective_value}"
print("Assignment problem OK - objective:", round(assign.objective_value, 4))

### Plot the solution

In [ ]:
nodes = i.toList()
edges, S = current_tours(x, nodes)
plot_solution(edges, 0, nodes, pos, S, title=f"Assignment solution: {len(S)} subtours!")

**What happened?** Every city has one way in and one way out, but the result is a set of small disconnected loops, called *subtours*. The assignment problem is only a **relaxation** of the TSP: it allows solutions that are not a single tour.

## 3. Subtour elimination with cuts

For every subtour $S$ we find, we add a cut that says at least one leg must leave $S$:

$$
\sum_{i \in S}\ \sum_{j \notin S} x_{ij} \;\ge\; 1
$$

We store the cuts in a coefficient table `cutcoeff[cut, i, j]` (1 if leg $i \to j$ leaves the subtour) and keep a dynamic set `allcuts` of the cuts added so far. The cut equation is indexed by `allcuts`, so a cut slot that is not used yet creates no constraint.

In [ ]:
cc = gp.Set(m, name="cc", records=[f"c{k}" for k in range(1, 1001)], description="cut slots")
curcut = gp.Set(m, name="curcut", domain=cc, description="current cut (one element)")
allcuts = gp.Set(m, name="allcuts", domain=cc, description="cuts added so far")
cutcoeff = gp.Parameter(m, name="cutcoeff", domain=[cc, i, j], description="1 if leg i->j leaves the subtour of this cut")

cut = gp.Equation(m, name="cut", domain=cc, description="subtour elimination cuts")
cut[allcuts] = gp.Sum([i, j], cutcoeff[allcuts, i, j] * x[i, j]) >= 1

tspcut = gp.Model(m, name="tspcut", equations=[rowsum, colsum, cut],
                  problem="MIP", sense="min", objective=objective)

**The cutting-plane loop.** In each iteration:

1. `current_tours(x, nodes)` reads the current solution and returns the legs used and `S`, a list of subtours.
2. If there is only one subtour, it visits every city, so it is a real tour and we stop.
3. Otherwise, each subtour gets its own cut. Every leg from a city inside the subtour to a city outside it gets coefficient 1 in the current cut slot.
4. The cut slot is switched on (`allcuts[curcut] = True`), `curcut` moves to the next slot, and the model is solved again.

In [ ]:
curcut[cc].where[gp.Ord(cc) == 1] = True   # start with slot c1
nodes_all = set(nodes)
history = []                                # (edges, subtours) of every iteration, for plotting

for it in range(100):
    edges, S = current_tours(x, nodes)
    history.append((edges, S))

    if len(S) == 1:                         # one tour through all cities: done
        break

    for s in S:
        inside = set(s.nodes())
        outside = nodes_all - inside
        for a in inside:
            for b in outside:
                cutcoeff[curcut, a, b] = 1  # leg a -> b leaves the subtour
        allcuts[curcut] = True              # switch this cut on
        curcut[cc] = curcut[cc - 1]         # move to the next cut slot

    tspcut.solve(options=gp.Options(relative_optimality_gap=0))
    print(f"iteration {it + 1}: {len(S)} subtours cut, total cuts = {len(allcuts.toList())}, "
          f"objective = {tspcut.objective_value:.4f}")

print("\nNo subtours remaining - optimal tour found!")
print("Tour length:", round(tspcut.objective_value, 4))

In [ ]:
# CHECK
assert abs(tspcut.objective_value - 19.5235) < 1e-3, f"got {tspcut.objective_value}"
print("Optimal TSP tour OK")

### Plot the iterations

In [ ]:
for it, (edges, S) in enumerate(history):
    plot_solution(edges, it, nodes, pos, S)

In [ ]:
# Print the tour in visiting order
succ = dict(current_tours(x, nodes)[0])
start = nodes[0]
tour, cur = [start], succ[start]
while cur != start:
    tour.append(cur)
    cur = succ[cur]
print(" -> ".join(tour + [start]))

**Discussion**

* How many cuts were needed? How does that compare with the $2^{n}$ possible subsets of cities?
* Adding only the constraints that are violated, when they are needed, is the core idea behind *branch-and-cut*.

## 4. Bonus: compact MTZ formulation

Instead of adding cuts one by one, the Miller-Tucker-Zemlin (MTZ) formulation rules out all subtours at once with about $n^2$ extra constraints. It adds a variable $u_i$, the **position of city $i$ in the tour** ($1 \le u_i \le n$).

**The idea.** If the tour goes from $i$ to $j$ ($x_{ij} = 1$), then $j$ comes right after $i$:

$$ x_{ij} = 1 \;\Rightarrow\; u_j \ge u_i + 1 $$

**As a big-M constraint.** We want this constraint only when $x_{ij} = 1$. When $x_{ij} = 0$ it must be switched off, so we relax it by a big number $M$:

$$ u_j \;\ge\; u_i + 1 - M\,(1 - x_{ij}) $$

* $x_{ij} = 1$: the $M$ term disappears and we get $u_j \ge u_i + 1$.
* $x_{ij} = 0$: we get $u_j \ge u_i + 1 - M$. With $M = n$ this always holds, because $u_i - u_j \le n - 1$ when every $u$ is between $1$ and $n$.

Choosing $M = n$ and moving everything to one side gives the usual MTZ form:

$$ u_i - u_j + n\, x_{ij} \;\le\; n - 1 \qquad \forall\, i \ne j,\;\; i, j \ne \text{start city} $$

**Why leave out the start city?** Along a subtour the positions would have to keep increasing all the way round, $u_a < u_b < \dots < u_a$, which is impossible. So every subtour is ruled out. The one real tour has to come back to its start, though, so we drop the constraint for every leg into or out of the start city.

The start city `first` and the position variable `u` are declared for you.

In [ ]:
n = len(nodes)
first = gp.Set(m, name="first", domain=i, records=[nodes[0]], description="start city")
u = gp.Variable(m, name="u", domain=i, description="position of city in the tour")

### TODO 9 (bonus) - Bounds on `u`
Every position is between $1$ and $n$.

*Hint:* `u.lo[i]` and `u.up[i]` set the lower and upper bound for every city.

In [ ]:
# TODO: your code here
u.lo[i] =
u.up[i] =

### TODO 10 (bonus) - The MTZ equation
* Declare the equation `mtz` over the domain `[i, j]` (see how `rowsum` was declared).
* Write the constraint $u_i - u_j + n\, x_{ij} \le n - 1$. The `.where[...]` condition is written for you: it leaves out the start city and the diagonal.

In [ ]:
# TODO: your code here
mtz = ...

mtz[i, j].where[(~first[i]) & (~first[j]) & (gp.Ord(i) != gp.Ord(j))] = ...

### TODO 11 (bonus) - The MTZ model
Build the model `tsp_mtz`: the assignment constraints plus `mtz`, with the same objective.

In [ ]:
# TODO: your code here
tsp_mtz = gp.Model(m, name="tsp_mtz", equations=..., problem=..., sense=..., objective=...)

In [ ]:
tsp_mtz.solve(options=gp.Options(relative_optimality_gap=0))
print("MTZ objective:", round(tsp_mtz.objective_value, 4),
      "| cutting-plane objective: 19.5235")

In [ ]:
edges, S = current_tours(x, nodes)
plot_solution(edges, 0, nodes, pos, S, title=f"MTZ solution (length {tsp_mtz.objective_value:.3f})")

## Take-aways

* GAMSPy lets you write the math almost 1:1: `Sum`, `where`, dynamic sets
* The **same container** holds several models (`assign`, `tspcut`, `tsp_mtz`) sharing variables and equations
* Python handles the algorithm logic (graph analysis with `networkx`); GAMSPy + the solver handle the optimization
* Iterative cuts vs. compact MTZ: two classic ways to kill subtours